# Incident Commander: TRL GRPO Training Notebook

This notebook is the judge-friendly training artifact for the OpenEnv hackathon submission. It runs a minimal Hugging Face TRL `GRPOTrainer` loop against the local Incident Commander environment and writes reward/loss plots to `outputs/evals/trl_grpo/`.

Why this notebook exists:
- It satisfies the official requirement for a runnable TRL/Unsloth training notebook.
- It trains against the actual environment instead of a static dataset.
- It keeps the run intentionally small so you can smoke-test it in Colab before scaling.

## 1. Runtime setup

Recommended Colab runtime: `T4 GPU` or better.

This notebook uses **HF TRL** directly, which satisfies the hackathon requirement. If you want to accelerate later, you can swap in an Unsloth-backed recipe once the baseline is stable.

In [1]:
!nvidia-smi || true

Sat Apr 25 13:45:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.79                 Driver Version: 595.79         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P0             10W /   75W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
%cd ..
%pip install -q --upgrade pip
%pip install -q -e ".[training]"

d:\hackathon\finale
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import trl
from trl.trainer.grpo_config import GRPOConfig

print('trl version:', trl.__version__)
print('GRPOConfig import OK:', GRPOConfig.__name__)

d:\hackathon\finale\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


trl version: 0.24.0
GRPOConfig import OK: GRPOConfig


## 2. Clone the repo

Replace the repo URL below if you are running from a fork.

In [5]:
from pathlib import Path
print('Working directory:', Path.cwd())
print('Repo root detected:', (Path.cwd() / 'examples' / 'trl_grpo_training.py').exists())

Working directory: d:\hackathon\finale
Repo root detected: True


## 3. Optional HF login

Only needed if you want to push trained adapters or use gated/private models.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4. Run a minimal GRPO training pass

This uses the local script `examples/trl_grpo_training.py`. The default model is `Qwen/Qwen2.5-0.5B-Instruct`, chosen to keep the notebook lightweight enough for a T4 smoke test.

If the run is stable, increase `--max-steps` and `--dataset-repeats` for a stronger curve.

> **Note on cell outputs vs committed artifacts:** The cell output below is from a local smoke-test run using `sshleifer/tiny-gpt2` (2 steps, ~19s) to verify the pipeline end-to-end. The committed artifacts in `outputs/evals/trl_grpo/` — `trl_grpo_run.json`, `trl_grpo_reward_curve.png`, `trl_grpo_loss_curve.png` — are from the full Qwen/Qwen2.5-0.5B-Instruct run (20 steps, seed 42, commit b3408d1, ~34 min CPU). To reproduce the full run: `python examples/trl_grpo_training.py --model Qwen/Qwen2.5-0.5B-Instruct --max-steps 20 --dataset-repeats 24`

In [6]:
import os
os.environ['PYTHONUTF8'] = '1'
!python -u examples/trl_grpo_training.py --model sshleifer/tiny-gpt2 --max-steps 2 --dataset-repeats 4

{'loss': -0.0, 'grad_norm': 0.00015445865574292839, 'learning_rate': 5e-06, 'num_tokens': 1794.0, 'completions/mean_length': 768.0, 'completions/min_length': 768.0, 'completions/max_length': 768.0, 'completions/clipped_ratio': 1.0, 'completions/mean_terminated_length': 0.0, 'completions/min_terminated_length': 0.0, 'completions/max_terminated_length': 0.0, 'rewards/reward_func/mean': -2.945000171661377, 'rewards/reward_func/std': 2.2980971336364746, 'reward': -2.945000171661377, 'reward_std': 2.2980971336364746, 'frac_reward_zero_std': 0.0, 'entropy': 10.82457447052002, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'epoch': 0.25}
{'loss': 0.0, 'grad_norm': 0.0001860943011706695, 'learning_rate': 2.5e-06, 'num_tokens': 3586.0, 'completions/mean_length': 768.0, 'completions/min_length': 768.0, 'completions/max_length': 768.0, 'completions/clipped_ratio': 1.0, 'completions/mean_terminated_leng

The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
d:\hackathon\finale\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.

  0%|          | 0/2 [00:00<?, ?it/s]d:\hackathon\finale\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

 50%|█████     | 1/2 [00:10<00:10, 10.16s/it]
                                             

100%|██████████| 2/2 [00:19<00:00,  9.52

## 5. Inspect generated artifacts

In [7]:
from pathlib import Path

artifact_dir = Path('outputs/evals/trl_grpo')
sorted(p.name for p in artifact_dir.glob('*'))

['.gitkeep',
 'trl_grpo_loss_curve.png',
 'trl_grpo_reward_curve.png',
 'trl_grpo_run.json',
 'trl_grpo_summary.json']

In [8]:
import json
from pathlib import Path

error_path = Path('outputs/evals/trl_grpo/trl_grpo_error.json')
if error_path.exists():
    print(json.dumps(json.loads(error_path.read_text()), indent=2))
else:
    print('No error artifact found.')

No error artifact found.


In [9]:
from pathlib import Path

artifact_dir = Path('outputs/evals/trl_grpo')
for name in ['trl_grpo_reward_curve.png', 'trl_grpo_loss_curve.png']:
    path = artifact_dir / name
    if path.exists():
        print(f'Found artifact: {path}')
    else:
        print(f'Missing artifact: {path}')

Found artifact: outputs\evals\trl_grpo\trl_grpo_reward_curve.png
Found artifact: outputs\evals\trl_grpo\trl_grpo_loss_curve.png


In [10]:
import json
from pathlib import Path

summary_path = Path('outputs/evals/trl_grpo/trl_grpo_run.json')
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))
else:
    print('Training summary not found. Check outputs/evals/trl_grpo/trl_grpo_error.json for the root cause.')

{
  "model_name": "sshleifer/tiny-gpt2",
  "seed": 42,
  "max_steps": 2,
  "dataset_repeats": 4,
  "train_runtime": 19.2365,
  "train_steps_per_second": 0.104,
  "reward_points": 2,
  "loss_points": 2,
  "final_reward": -4.399999618530273,
  "final_loss": 0.0,
  "best_reward": -2.945000171661377
}


## 6. Suggested next scaling steps

After the smoke test succeeds:
- Increase `--max-steps` to `25` or `50`
- Increase `--dataset-repeats` to `24` or `48`
- Compare different small instruct models
- Save the best resulting plots back into the repo and link them from the README

## Notes

- This notebook uses **HF TRL**, which is explicitly allowed by the judging materials.
- Unsloth can be used later for acceleration, but the priority is a stable, reproducible end-to-end run.
- The environment logic remains the same one used by the demo and evaluation scripts, so this training artifact stays aligned with the main submission.